# Social Backlash Risk Prototype for SNS Posts

This notebook is a prototype for lightly estimating whether a post may be **criticized, misunderstood, or receive backlash** in a public SNS context. It is not a **toxicity detection** notebook.

## Step-by-Step Pipeline
1. User input
2. Preprocessing
3. Tokenization
4. Embedding
5. First-pass risk classification (semantic similarity + surface signals)
6. Second-pass contextual analysis (explanations + prompt draft)
7. Safer rewrite suggestion
8. Final result output

## Design Notes
- It handles slang, emojis, repeated characters, and short impulsive sentences.
- Even without profanity, sarcasm, overgeneralization, absolute phrasing, and public-context inappropriateness are treated as risk signals.
- Because this is a classroom prototype, it prioritizes **explainability** and **clear structure** over heavy modeling.

In [ ]:
# Run this only once if needed.
# If the packages are missing in VSCode / Jupyter, uncomment the line below.

# %pip install -q pandas scikit-learn sentence-transformers


In [2]:
import math
import re
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMER_AVAILABLE = True
except ImportError:
    SentenceTransformer = None
    SENTENCE_TRANSFORMER_AVAILABLE = False

pd.set_option("display.max_colwidth", 120)

EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001FAD6"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F900-\U0001F9FF"
    "]",
    flags=re.UNICODE,
)

IMPORTANT_EMOJIS = {
    "lol", "lmao", "rofl", "smh", "ikr", "fr", "tbh", "wtf", "omg",
    "ㅋㅋ", "ㅎㅎ", "ㅠㅠ", "ㅜㅜ", "^^", ":)", ":(", "lolol",
    "😂", "🤣", "😒", "🙄", "😡", "🤮", "🤡", "💀"
}

HEDGE_PATTERNS = [
    r"\bi think\b", r"\bi feel\b", r"\bmaybe\b", r"\bperhaps\b", r"\bit seems\b", r"\bmight\b"
]

PUBLIC_CONTEXT_HINTS = {"public", "instagram", "x", "community", "forum", "twitter", "tiktok"}
VAGUE_TARGET_HINTS = {"they", "them", "people", "this", "that", "those", "everyone"}

DIMENSION_PROTOTYPES = {
    "Aggression": [
        "This sounds openly hostile and harsh.",
        "The post attacks the target in a blunt and aggressive way.",
        "This wording feels insulting and confrontational."
    ],
    "Group Generalization": [
        "The post makes a sweeping claim about a whole group.",
        "This sounds like a broad generalization about many people.",
        "The wording treats a group as if everyone in it is the same."
    ],
    "Sarcasm / Mockery": [
        "The tone sounds sarcastic, dismissive, or mocking.",
        "This post seems written to ridicule someone.",
        "The wording has a sneering or eye-rolling tone."
    ],
    "Overconfident Judgment": [
        "The post presents a personal opinion like an unquestionable fact.",
        "This sounds overly certain and leaves little room for nuance.",
        "The wording feels absolute rather than tentative."
    ],
    "Context Inappropriateness": [
        "This could feel inappropriate in a public social media setting.",
        "The same statement may draw criticism when posted publicly.",
        "This wording may not fit a broad online audience."
    ],
    "Misinterpretability": [
        "The message is easy to misunderstand and may sound harsher than intended.",
        "This post is ambiguous enough that readers may infer a hostile meaning.",
        "The target or intent is unclear, which raises the chance of misreading."
    ],
    "Norm Violation": [
        "The tone risks sounding demeaning, dehumanizing, or socially unacceptable.",
        "This wording may cross a social norm boundary.",
        "The post could be read as attacking someone's dignity or status."
    ],
}

DIMENSION_WEIGHTS = {
    "Aggression": 1.3,
    "Group Generalization": 1.2,
    "Sarcasm / Mockery": 1.15,
    "Overconfident Judgment": 1.0,
    "Context Inappropriateness": 1.15,
    "Misinterpretability": 1.0,
    "Norm Violation": 1.4,
}

DIMENSION_DESCRIPTIONS = {
    "Aggression": "Level of direct aggression or harsh criticism",
    "Group Generalization": "Degree of broad judgment about an entire group",
    "Sarcasm / Mockery": "Intensity of sarcasm, mockery, or derision",
    "Overconfident Judgment": "Degree to which a personal opinion is framed like an absolute fact",
    "Context Inappropriateness": "Likelihood of appearing inappropriate in a public SNS context",
    "Misinterpretability": "Likelihood that the intent could be distorted or read as hostile",
    "Norm Violation": "Tendency toward discriminatory, dehumanizing, or norm-violating language",
}


In [1]:
import sys
print(sys.executable)

/Users/yooseungjong/Desktop/Natural_Lang_Proj/.venv/bin/python


In [3]:
@dataclass
class PipelineOutput:
    context: str
    original_text: str
    normalized_text: str
    tokens: List[str]
    embedding_model: str
    embedding_preview: List[float]
    risk_dimensions: Dict[str, int]
    risk_score: int
    issue_categories: List[str]
    explanation_points: List[str]
    llm_prompt: str
    rewrite_suggestion: str


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def preserve_and_clean_special_characters(text: str) -> str:
    text = text.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = re.sub(r"[^\w\s!?.,'@#:/()😂🤣😒🙄😡🤮🤡💀ㅠㅜㅎㅋ^~+-]", " ", text)
    text = re.sub(r"([!?.,~]){3,}", r"\1\1", text)
    return normalize_whitespace(text)


def reduce_repeated_characters(text: str, keep: int = 2) -> str:
    def _reduce(match: re.Match) -> str:
        char = match.group(1)
        return char * keep

    return re.sub(r"(.)\1{2,}", _reduce, text)


def preprocess_text(text: str) -> str:
    text = normalize_whitespace(text)
    text = preserve_and_clean_special_characters(text)
    text = reduce_repeated_characters(text)
    return text


def tokenize_text(text: str) -> List[str]:
    token_pattern = r"#[\w_]+|@[\w_]+|[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[가-힣]+|[ㅋㅎㅠㅜ]+|[!?.,]+|😂|🤣|😒|🙄|😡|🤮|🤡|💀"
    return re.findall(token_pattern, text.lower())


class LightweightEmbedder:
    def __init__(self, seed_texts: List[str] = None):
        prototype_texts = [item for values in DIMENSION_PROTOTYPES.values() for item in values]
        self.seed_texts = seed_texts or [
            "this place is trash lol",
            "I did not really enjoy this place",
            "why are people always like this",
            "I disagree but I may be wrong",
            "everyone here is so annoying",
            "the food was not my style"
        ] + prototype_texts
        self.model_name = "tfidf-fallback"
        self.tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        self.tfidf.fit(self.seed_texts)
        self.sentence_model = None

        if SENTENCE_TRANSFORMER_AVAILABLE:
            try:
                self.sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
                self.model_name = "sentence-transformers/all-MiniLM-L6-v2"
            except Exception:
                self.sentence_model = None

    def encode(self, text: str) -> np.ndarray:
        if self.sentence_model is not None:
            return np.asarray(self.sentence_model.encode(text), dtype=float)
        return self.tfidf.transform([text]).toarray()[0]


def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    norm_a = float(np.linalg.norm(vec_a))
    norm_b = float(np.linalg.norm(vec_b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))


def quantize_signal(value: float, low: float = 0.35, high: float = 0.5) -> int:
    if value >= high:
        return 2
    if value >= low:
        return 1
    return 0


class SemanticRiskScorer:
    def __init__(self, embedder: LightweightEmbedder):
        self.embedder = embedder
        self.prototype_embeddings = {
            name: [self.embedder.encode(example) for example in examples]
            for name, examples in DIMENSION_PROTOTYPES.items()
        }

    def prototype_score(self, text_embedding: np.ndarray, dimension: str) -> float:
        similarities = [
            cosine_similarity(text_embedding, prototype_embedding)
            for prototype_embedding in self.prototype_embeddings[dimension]
        ]
        return max(similarities) if similarities else 0.0


def measure_surface_signals(text: str, tokens: List[str], context: str) -> Dict[str, float]:
    lowered = text.lower()
    exclamations = min(text.count("!"), 3) / 3
    question_bursts = min(text.count("?"), 3) / 3
    repeated_punctuation = 1.0 if re.search(r"([!?.,~])\1", text) else 0.0
    emoji_density = min(sum(token in IMPORTANT_EMOJIS for token in tokens), 3) / 3
    hedge_density = min(sum(bool(re.search(pattern, lowered)) for pattern in HEDGE_PATTERNS), 2) / 2
    public_context = 1.0 if any(hint in context.lower() for hint in PUBLIC_CONTEXT_HINTS) else 0.0
    short_post = 1.0 if len(tokens) <= 6 else 0.0
    vague_target = 1.0 if any(token in VAGUE_TARGET_HINTS for token in tokens) else 0.0

    return {
        "exclamations": exclamations,
        "question_bursts": question_bursts,
        "repeated_punctuation": repeated_punctuation,
        "emoji_density": emoji_density,
        "hedge_density": hedge_density,
        "public_context": public_context,
        "short_post": short_post,
        "vague_target": vague_target,
    }


def score_risk_dimensions(
    text: str,
    tokens: List[str],
    context: str,
    embedding_vector: np.ndarray,
    scorer: SemanticRiskScorer,
) -> Dict[str, int]:
    semantic_scores = {
        name: scorer.prototype_score(embedding_vector, name)
        for name in DIMENSION_PROTOTYPES
    }
    surface = measure_surface_signals(text, tokens, context)

    aggression_signal = semantic_scores["Aggression"] + 0.12 * surface["exclamations"] + 0.08 * surface["repeated_punctuation"]
    group_signal = semantic_scores["Group Generalization"] + 0.10 * surface["vague_target"]
    sarcasm_signal = semantic_scores["Sarcasm / Mockery"] + 0.10 * surface["emoji_density"] + 0.05 * surface["question_bursts"]
    overconfidence_signal = semantic_scores["Overconfident Judgment"] - 0.12 * surface["hedge_density"]
    context_signal = semantic_scores["Context Inappropriateness"] + 0.12 * surface["public_context"]
    misread_signal = semantic_scores["Misinterpretability"] + 0.10 * surface["short_post"] + 0.10 * surface["vague_target"]
    norm_signal = semantic_scores["Norm Violation"] + 0.05 * surface["public_context"]

    return {
        "Aggression": quantize_signal(aggression_signal),
        "Group Generalization": quantize_signal(group_signal),
        "Sarcasm / Mockery": quantize_signal(sarcasm_signal),
        "Overconfident Judgment": quantize_signal(overconfidence_signal),
        "Context Inappropriateness": quantize_signal(context_signal),
        "Misinterpretability": quantize_signal(misread_signal),
        "Norm Violation": quantize_signal(norm_signal),
    }


def calculate_final_risk_score(risk_dimensions: Dict[str, int]) -> int:
    weighted_sum = sum(risk_dimensions[name] * weight for name, weight in DIMENSION_WEIGHTS.items())
    max_sum = sum(2 * weight for weight in DIMENSION_WEIGHTS.values())
    score = round((weighted_sum / max_sum) * 100)
    return int(np.clip(score, 0, 100))


def derive_issue_categories(risk_dimensions: Dict[str, int]) -> List[str]:
    issue_categories = [name for name, value in risk_dimensions.items() if value >= 1]
    return issue_categories or ["Low explicit risk"]


def build_analysis_prompt(context: str, original_text: str, risk_dimensions: Dict[str, int]) -> str:
    dimension_lines = "\n".join(f"- {name}: {score}" for name, score in risk_dimensions.items())
    return f"""You are analyzing social backlash risk for a public online post.

Context: {context}
Original text: {original_text}

Risk dimension scores:
{dimension_lines}

Explain why this post may be criticized, misunderstood, or receive backlash.
Focus on sarcasm, dismissiveness, generalization, public-context appropriateness, and possible audience reaction.
Then suggest a safer rewrite that preserves the user's intent.
""".strip()


def generate_explanations(text: str, tokens: List[str], context: str, risk_dimensions: Dict[str, int]) -> List[str]:
    reasons = []

    if risk_dimensions["Aggression"] >= 1:
        reasons.append("The wording may sound somewhat aggressive or overly harsh, which can make the target react defensively.")
    if risk_dimensions["Sarcasm / Mockery"] >= 1:
        reasons.append("It may read as sarcastic or mocking, so people could interpret it as colder than intended.")
    if risk_dimensions["Group Generalization"] >= 1:
        reasons.append("It can sound like it is judging an entire group rather than a specific case, which may trigger pushback.")
    if risk_dimensions["Overconfident Judgment"] >= 1:
        reasons.append("It may come across as a definitive judgment rather than a personal opinion, which can invite disagreement or arguments.")
    if risk_dimensions["Context Inappropriateness"] >= 1:
        reasons.append(f"In a public context like {context}, the same wording may come across as ruder or less appropriate.")
    if risk_dimensions["Misinterpretability"] >= 1:
        reasons.append("Because the sentence is short or the target is vague, it may be misread as an attack on a specific person or group.")
    if risk_dimensions["Norm Violation"] >= 1:
        reasons.append("It may be interpreted as violating social norms or carrying a discriminatory tone, which can lead to stronger backlash.")

    if not reasons:
        reasons.append("The post is not strongly aggressive on its face, but readers may still interpret it differently depending on context.")

    return reasons


def soften_expression(text: str) -> str:
    softened = preserve_and_clean_special_characters(text)
    softened = re.sub(r"\s+", " ", softened).strip(" .!?")

    if not softened:
        return "It may be better to rephrase this in a more neutral way."

    softened = softened[0].lower() + softened[1:] if len(softened) > 1 else softened.lower()

    if not re.search(r"\b(I|i)\b", softened):
        softened = f"I think {softened}"

    if not softened.endswith("."):
        softened += "."

    return softened[0].upper() + softened[1:]


def generate_rewrite_suggestion(original_text: str, risk_dimensions: Dict[str, int]) -> str:
    softened = soften_expression(original_text)

    if risk_dimensions["Aggression"] + risk_dimensions["Sarcasm / Mockery"] >= 3:
        softened = softened.replace("I think", "From my perspective,")

    if risk_dimensions["Overconfident Judgment"] >= 1 and "just my impression" not in softened:
        softened = softened.rstrip(".") + ", but that's just my impression."

    if risk_dimensions["Group Generalization"] >= 1 and "in this case" not in softened:
        softened = softened.rstrip(".") + " in this case."

    return softened


def run_backlash_risk_pipeline(text: str, context: str = "Instagram / Public") -> PipelineOutput:
    normalized_text = preprocess_text(text)
    tokens = tokenize_text(normalized_text)

    embedder = LightweightEmbedder()
    embedding_vector = embedder.encode(normalized_text)
    semantic_scorer = SemanticRiskScorer(embedder)

    risk_dimensions = score_risk_dimensions(normalized_text, tokens, context, embedding_vector, semantic_scorer)
    risk_score = calculate_final_risk_score(risk_dimensions)
    issue_categories = derive_issue_categories(risk_dimensions)
    explanation_points = generate_explanations(normalized_text, tokens, context, risk_dimensions)
    llm_prompt = build_analysis_prompt(context, text, risk_dimensions)
    rewrite_suggestion = generate_rewrite_suggestion(text, risk_dimensions)

    embedding_preview = [round(float(value), 4) for value in embedding_vector[:8]]

    return PipelineOutput(
        context=context,
        original_text=text,
        normalized_text=normalized_text,
        tokens=tokens,
        embedding_model=embedder.model_name,
        embedding_preview=embedding_preview,
        risk_dimensions=risk_dimensions,
        risk_score=risk_score,
        issue_categories=issue_categories,
        explanation_points=explanation_points,
        llm_prompt=llm_prompt,
        rewrite_suggestion=rewrite_suggestion,
    )


def display_pipeline_result(result: PipelineOutput) -> None:
    print(f"Context: {result.context}")
    print(f"Input: {result.original_text}")
    print(f"Normalized: {result.normalized_text}")
    print(f"Tokens: {result.tokens}")
    print(f"Embedding model: {result.embedding_model}")
    print(f"Embedding preview: {result.embedding_preview}")
    print()
    print(f"Risk Score: {result.risk_score}/100")
    print("Risk Dimensions:")
    for name, score in result.risk_dimensions.items():
        print(f"- {name}: {score} ({DIMENSION_DESCRIPTIONS[name]})")
    print()
    print("Issue Categories:")
    for category in result.issue_categories:
        print(f"- {category}")
    print()
    print("Why risky:")
    for reason in result.explanation_points:
        print(f"- {reason}")
    print()
    print(f"Safer rewrite: {result.rewrite_suggestion}")


## Step-by-Step Example Run

The cell below runs the full pipeline on one sample input. For a class presentation, you can either call each function in separate cells or show only the final integrated function.

In [ ]:
sample_context = "Instagram / Public"
sample_text = "This place is trash lol"

result = run_backlash_risk_pipeline(sample_text, context=sample_context)
display_pipeline_result(result)


In [ ]:
# Run the cell below if you want to view the result as a table.

result_table = pd.DataFrame(
    {
        "Dimension": list(result.risk_dimensions.keys()),
        "Score": list(result.risk_dimensions.values()),
        "Description": [DIMENSION_DESCRIPTIONS[name] for name in result.risk_dimensions.keys()],
    }
)

result_table


In [ ]:
# Prompt draft for second-pass contextual analysis
# You can later connect this to the OpenAI API or another LLM.

print(result.llm_prompt)


## Short Pipeline Explanation

### 1. Input
The user enters a sentence they are planning to post on social media.

### 2. Preprocessing
- Normalize whitespace
- Clean special characters
- Reduce excessive repeated characters
- Preserve emotionally meaningful emojis and slang where possible

### 3. Tokenization
Split English words, Korean text, hashtags, mentions, emojis, and punctuation using simple rule-based logic.

### 4. Embedding
- Use sentence embeddings if `sentence-transformers` is installed
- Otherwise fall back to a `TF-IDF`-based representation

### 5. First-Pass Risk Classification
Compute seven dimension scores from semantic similarity to prototype examples plus lightweight surface-level signals, then combine them into a final weighted risk score.

### 6. Second-Pass Contextual Analysis
Generate explainable reasons for the risk score and build a prompt draft that can later be sent to an LLM.

### 7. Rewrite Suggestion
Soften aggressive wording, absolutes, and mocking tone while preserving the original intent as much as possible.

## Future Improvement Ideas

1. Build a real labeled dataset
   - Define labels specifically for `social backlash risk`, not generic `toxicity`

2. Add a lightweight classifier
   - The current version is mostly rule-based, so a simple classifier such as `LogisticRegression` or `LinearSVC` could be added later.

3. Expand contextual features
   - Platform type (Instagram, X, email)
   - Relationship to the audience (friends, general public, workplace)
   - Visibility scope (fully public, story, community)

4. Expand multilingual support
   - Build stronger lexicons for Korean slang, English slang, and mixed-language sentences

5. Connect an LLM for second-pass analysis
   - The current `llm_prompt` can be used to improve both explanation quality and rewrite quality.

6. Improve user-facing output
   - A UI that shows not only `why it is risky` but also `how to fix it` more concretely would make the demo stronger for presentation.